# TASTERA — Movie Recommendation Baselines

This notebook compares rating-prediction baselines and top-K movie ranking methods on MovieLens Latest Small.

**Methods**
- Global-mean and user-mean rating predictors, evaluated with MAE and RMSE.
- Popularity, genre-content, and item-item collaborative-filtering rankings.

Rating-prediction scores and ranking metrics measure different tasks and are reported separately.


In [1]:
from pathlib import Path
import sys
import scipy
import sklearn
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

def find_project_root(start: Path) -> Path:
    """Find the repository root without relying on a machine-specific path."""
    for candidate in (start, *start.parents):
        if (candidate / "src" / "tastera" / "recommenders" / "baselines.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the TASTERA repository root. "
        "Run this notebook from inside the cloned repository."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "movielens" / "ml-latest-small"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

if not (DATA_DIR / "ratings.csv").exists():
    raise FileNotFoundError(f"Ratings file not found: {DATA_DIR / 'ratings.csv'}")

if not (DATA_DIR / "movies.csv").exists():
    raise FileNotFoundError(f"Movies file not found: {DATA_DIR / 'movies.csv'}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from tastera.recommenders.baselines import (
    ContentBasedRecommender,
    GlobalMeanRecommender,
    PopularityRecommender,
    UserMeanRecommender,
)

movies = pd.read_csv(DATA_DIR / "movies.csv")
ratings = pd.read_csv(DATA_DIR / "ratings.csv")

print("Project root:", PROJECT_ROOT)
print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__, "pandas:", pd.__version__)
print("SciPy:", scipy.__version__, "scikit-learn:", sklearn.__version__)


Project root: D:\Tastera\Tastera
Movies: (9742, 3)
Ratings: (100836, 4)
Python: 3.13.9
NumPy: 2.3.5 pandas: 2.3.3
SciPy: 1.16.3 scikit-learn: 1.7.2


## 1. Validate and split the data

A chronological holdout is used: the 80th-percentile timestamp is the cutoff. Training contains timestamps strictly below the cutoff; test contains timestamps at or after it. This keeps evaluation ratings out of model fitting. Because the split is timestamp-based, some users or movies may be absent from training; the user-mean model falls back to the global mean for unseen users.


In [2]:
required_rating_cols = {"userId", "movieId", "rating", "timestamp"}
required_movie_cols = {"movieId", "title", "genres"}
assert required_rating_cols.issubset(ratings.columns), required_rating_cols - set(ratings.columns)
assert required_movie_cols.issubset(movies.columns), required_movie_cols - set(movies.columns)
assert ratings["rating"].between(0.5, 5.0).all()
assert (ratings["rating"].mul(2) % 1 == 0).all(), "Expected half-star rating increments"

cutoff = ratings["timestamp"].quantile(0.80)
train = ratings.loc[ratings["timestamp"] < cutoff].copy()
test = ratings.loc[ratings["timestamp"] >= cutoff].copy()

if train.empty or test.empty:
    raise ValueError("Temporal split produced an empty train or test set.")

print(f"Cutoff timestamp: {cutoff}")
print(f"Train rows: {len(train):,} ({len(train)/len(ratings):.1%})")
print(f"Test rows:  {len(test):,} ({len(test)/len(ratings):.1%})")
print(f"Train time range: {train.timestamp.min()} to {train.timestamp.max()}")
print(f"Test time range:  {test.timestamp.min()} to {test.timestamp.max()}")


Cutoff timestamp: 1458635171.0
Train rows: 80,668 (80.0%)
Test rows:  20,168 (20.0%)
Train time range: 828124615 to 1458635162
Test time range:  1458635171 to 1537799250


## 2. Rating-prediction baselines

MAE measures average absolute error; RMSE penalizes larger errors more strongly. Lower values are better. Both models use only the training split.

In [3]:
rating_models = {
    "global_mean": GlobalMeanRecommender().fit(train),
    "user_mean": UserMeanRecommender().fit(train),
}

metric_rows = []
predictions = {}
for model_name, model in rating_models.items():
    y_pred = model.predict(test["userId"], test["movieId"])
    predictions[model_name] = y_pred
    metric_rows.append({
        "model": model_name,
        "test_rows": len(test),
        "MAE": mean_absolute_error(test["rating"], y_pred),
        "RMSE": np.sqrt(mean_squared_error(test["rating"], y_pred)),
    })

rating_metrics = pd.DataFrame(metric_rows).sort_values("RMSE").reset_index(drop=True)
display(rating_metrics)
rating_metrics.to_csv(OUTPUT_DIR / "baseline_rating_metrics.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'baseline_rating_metrics.csv'}")


,model,test_rows,MAE,RMSE
0,user_mean,20168,0.839610,1.064709
1,global_mean,20168,0.854019,1.078712


Saved: D:\Tastera\Tastera\data\processed\baseline_rating_metrics.csv


## 3. Popularity recommendations

The ranking is fitted on training ratings only. The minimum count is configurable; 20 is a reasonable initial threshold for this dataset, not a universal optimum.

In [4]:
popularity_model = PopularityRecommender().fit(train, movies, min_ratings=20)
popular = popularity_model.recommend(n=20)
display(popular[["movieId", "title", "genres", "rating_count", "mean_rating"]])
popular.to_csv(OUTPUT_DIR / "baseline_popularity_top20.csv", index=False)


,movieId,title,genres,rating_count,mean_rating
0,356,Forrest Gump (1994),Comedy|Drama|Romance|War,267,4.166667
1,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,264,4.189394
2,318,"Shawshank Redemption, The (1994)",Crime|Drama,256,4.423828
3,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,235,4.202128
4,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,219,4.214612
5,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,218,3.786697
6,110,Braveheart (1995),Action|Drama|War,212,4.058962
7,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,206,4.264563
8,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi,200,3.970000
9,527,Schindler's List (1993),Drama|War,188,4.228723


## 4. Genre-content recommendations

The example user is selected from the training data so the demonstration is reproducible. The model uses training ratings to identify liked movies and excludes movies the user has already rated in training.

In [5]:
example_user_id = int(train["userId"].value_counts().index[0])
content_model = ContentBasedRecommender().fit(movies)
content_recs = content_model.recommend_for_user(
    example_user_id, train, n=20, like_threshold=4.0
)

print(f"Example user ID: {example_user_id}")
print("Highly rated training movies for this user:")
liked_ids = train.loc[
    (train["userId"] == example_user_id) & (train["rating"] >= 4.0), "movieId"
]
display(movies[movies["movieId"].isin(liked_ids)][["title", "genres"]].head(15))
print("Recommended unseen movies:")
display(content_recs[["movieId", "title", "genres", "content_score", "matched_genres"]])
content_recs.to_csv(OUTPUT_DIR / "baseline_content_top20.csv", index=False)


Example user ID: 414
Highly rated training movies for this user:


,title,genres
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,Grumpier Old Men (1995),Comedy|Romance
10,"American President, The (1995)",Comedy|Drama|Romance
16,Sense and Sensibility (1995),Drama|Romance
20,Get Shorty (1995),Comedy|Crime|Thriller
31,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
32,Babe (1995),Children|Drama
35,Clueless (1995),Comedy|Romance
43,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
46,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


Recommended unseen movies:


,movieId,title,genres,content_score,matched_genres
0,117867,'71 (2014),Action|Drama|Thriller|War,1.0,"Action, Drama, Thriller, War"
1,97757,'Hellboy': The Seeds of Creation (2004),Action|Adventure|Comedy|Documentary|Fantasy,1.0,"Action, Adventure, Comedy, Documentary, Fantasy"
2,26564,'Round Midnight (1986),Drama|Musical,1.0,"Drama, Musical"
3,27751,'Salem's Lot (2004),Drama|Horror|Mystery|Thriller,1.0,"Drama, Horror, Mystery, Thriller"
4,779,'Til There Was You (1997),Drama|Romance,1.0,"Drama, Romance"
5,149380,'Tis the Season for Love (2015),Romance,1.0,Romance
6,3112,'night Mother (1986),Drama,1.0,Drama
7,8169,*batteries not included (1987),Children|Comedy|Fantasy|Sci-Fi,1.0,"Children, Comedy, Fantasy, Sci-Fi"
8,5706,...All the Marbles (1981),Comedy|Drama,1.0,"Comedy, Drama"
9,3420,...And Justice for All (1979),Drama|Thriller,1.0,"Drama, Thriller"


## 5. Leakage-safe top-K ranking evaluation

For ranking evaluation, sort each user's interactions by timestamp and hold out that user's latest 20% (at least one interaction) as test data. Earlier interactions remain in training. A test interaction is relevant when its held-out rating is at least 4.0. Evaluate users with at least five training ratings, at least one training rating of 4.0 or higher (so a taste profile can be formed), and at least one relevant test interaction. Recommendations exclude every item rated by that user in training. Popularity and item-item similarity are fitted from ranking-training ratings only; genre matching uses the user's ranking-training likes and movie genre metadata. Test ratings are never used to fit a recommender. The rating-prediction section above keeps its global chronological split and reports a separate task.

Precision@10 divides hits by 10 even when a method returns fewer than 10 candidates; Recall@10 divides by the user's full set of relevant test items; NDCG@10 uses binary relevance with logarithmic rank discount. Catalog coverage is the share of MovieLens titles appearing in at least one evaluated top-10 list. These metrics answer ranking questions and should not be compared with MAE/RMSE.


In [6]:
from tastera.recommenders import ItemItemCollaborativeRecommender

RANK_K = 10
MIN_TRAIN_RATINGS = 5
RELEVANCE_THRESHOLD = 4.0

ranked_ratings = ratings.sort_values(["userId", "timestamp", "movieId"], kind="mergesort").copy()
per_user_holdout = np.ceil(ranked_ratings.groupby("userId").size() * 0.20).astype(int).clip(lower=1)
ranked_ratings["reverse_order"] = ranked_ratings.groupby("userId").cumcount(ascending=False)
ranked_ratings["holdout_size"] = ranked_ratings["userId"].map(per_user_holdout)
ranking_test = ranked_ratings.loc[ranked_ratings["reverse_order"] < ranked_ratings["holdout_size"]].drop(columns=["reverse_order", "holdout_size"])
ranking_train = ranked_ratings.loc[ranked_ratings["reverse_order"] >= ranked_ratings["holdout_size"]].drop(columns=["reverse_order", "holdout_size"])

train_counts = ranking_train.groupby("userId").size()
train_likes = ranking_train.loc[ranking_train["rating"] >= RELEVANCE_THRESHOLD].groupby("userId").size()
relevant_test = ranking_test.loc[ranking_test["rating"] >= RELEVANCE_THRESHOLD]
relevant_by_user = relevant_test.groupby("userId")["movieId"].apply(set).to_dict()
eligible_users = sorted(
    user_id for user_id, relevant_items in relevant_by_user.items()
    if train_counts.get(user_id, 0) >= MIN_TRAIN_RATINGS
    and train_likes.get(user_id, 0) >= 1
    and relevant_items
)

print(f"Ranking split: each user’s latest 20% of interactions held out")
print(f"Ranking train rows: {len(ranking_train):,}; ranking test rows: {len(ranking_test):,}")
print(f"Relevant test rule: rating >= {RELEVANCE_THRESHOLD}")
print(f"Eligible users: {len(eligible_users):,} of {ratings['userId'].nunique():,}")
print(f"Relevant held-out interactions: {len(relevant_test):,}")

item_item_model = ItemItemCollaborativeRecommender().fit(ranking_train, movies)
popularity_ranking = PopularityRecommender().fit(ranking_train, movies, min_ratings=20)
content_ranking = ContentBasedRecommender().fit(movies)

recommendation_lists = {"popularity": {}, "genre_content": {}, "item_item_cf": {}}
train_seen_by_user = ranking_train.groupby("userId")["movieId"].apply(set).to_dict()
for user_id in eligible_users:
    seen = train_seen_by_user.get(user_id, set())
    recommendation_lists["popularity"][user_id] = popularity_ranking.recommend(
        n=RANK_K, exclude_movie_ids=seen
    )["movieId"].tolist()
    recommendation_lists["genre_content"][user_id] = content_ranking.recommend_for_user(
        user_id, ranking_train, n=RANK_K, like_threshold=RELEVANCE_THRESHOLD
    )["movieId"].tolist()
    recommendation_lists["item_item_cf"][user_id] = item_item_model.recommend_for_user(
        user_id, n=RANK_K
    )["movieId"].tolist()

def ranking_metrics(recommendations, relevant_by_user, k=RANK_K):
    precision, recall, ndcg = [], [], []
    recommended_catalog = set()
    for user_id, items in recommendations.items():
        ranked = items[:k]
        relevant = relevant_by_user[user_id]
        hits = [1 if item in relevant else 0 for item in ranked]
        precision.append(sum(hits) / k)
        recall.append(sum(hits) / len(relevant))
        dcg = sum(hit / np.log2(rank + 2) for rank, hit in enumerate(hits))
        ideal_hits = min(k, len(relevant))
        idcg = sum(1 / np.log2(rank + 2) for rank in range(ideal_hits))
        ndcg.append(dcg / idcg if idcg else 0.0)
        recommended_catalog.update(ranked)
    return {
        f"Precision@{k}": float(np.mean(precision)),
        f"Recall@{k}": float(np.mean(recall)),
        f"NDCG@{k}": float(np.mean(ndcg)),
        f"Catalog coverage@{k}": len(recommended_catalog) / len(movies),
        "Users evaluated": len(recommendations),
    }

ranking_results = pd.DataFrame([
    {"model": name, **ranking_metrics(lists, relevant_by_user)}
    for name, lists in recommendation_lists.items()
]).sort_values(f"NDCG@{RANK_K}", ascending=False).reset_index(drop=True)
display(ranking_results)
ranking_results.to_csv(OUTPUT_DIR / "baseline_ranking_metrics.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'baseline_ranking_metrics.csv'}")


Ranking split: each user’s latest 20% of interactions held out


Ranking train rows: 80,419; ranking test rows: 20,417
Relevant test rule: rating >= 4.0
Eligible users: 590 of 610
Relevant held-out interactions: 9,509


,model,Precision@10,Recall@10,NDCG@10,Catalog coverage@10,Users evaluated
0,popularity,0.058136,0.048990,0.075633,0.010060,590
1,item_item_cf,0.037627,0.036585,0.048454,0.168754,590
2,genre_content,0.001356,0.001165,0.001439,0.005748,590


Saved: D:\Tastera\Tastera\data\processed\baseline_ranking_metrics.csv


## 6. Tune using a separate validation period

Do not tune parameters on the previously reported final ranking holdout. Split the ranking-training period again per user: the latest 20% of each user's training interactions become validation data, and the earlier 80% become tuning-training data. Use only that split to compare count-based popularity, Bayesian-smoothed popularity, and item-item collaborative filtering with different co-rating shrinkage. The selection metric is NDCG@10; also inspect Recall@10, catalog coverage, and intra-list genre diversity. A later final assessment should use genuinely unseen interactions collected after this experiment; the earlier holdout has already been examined.


In [7]:
# Split the prior ranking-training period again; keep ranking_test out of tuning.
tuning_order = ranking_train.sort_values(["userId", "timestamp", "movieId"], kind="mergesort").copy()
tuning_holdout = np.ceil(tuning_order.groupby("userId").size() * 0.20).astype(int).clip(lower=1)
tuning_order["reverse_order"] = tuning_order.groupby("userId").cumcount(ascending=False)
tuning_order["holdout_size"] = tuning_order["userId"].map(tuning_holdout)
validation = tuning_order.loc[tuning_order["reverse_order"] < tuning_order["holdout_size"]].drop(columns=["reverse_order", "holdout_size"])
tuning_train = tuning_order.loc[tuning_order["reverse_order"] >= tuning_order["holdout_size"]].drop(columns=["reverse_order", "holdout_size"])

validation_counts = tuning_train.groupby("userId").size()
validation_likes = tuning_train.loc[tuning_train["rating"] >= RELEVANCE_THRESHOLD].groupby("userId").size()
validation_relevant = validation.loc[validation["rating"] >= RELEVANCE_THRESHOLD]
validation_relevant_by_user = validation_relevant.groupby("userId")["movieId"].apply(set).to_dict()
validation_users = sorted(
    user_id for user_id, relevant_items in validation_relevant_by_user.items()
    if validation_counts.get(user_id, 0) >= MIN_TRAIN_RATINGS
    and validation_likes.get(user_id, 0) >= 1
    and relevant_items
)
validation_seen = tuning_train.groupby("userId")["movieId"].apply(set).to_dict()
genres_by_movie = {
    int(row.movieId): set(g for g in str(row.genres).split("|") if g and g != "(no genres listed)")
    for row in movies.itertuples(index=False)
}

print(f"Tuning train rows: {len(tuning_train):,}; validation rows: {len(validation):,}")
print(f"Validation users: {len(validation_users):,}; relevant interactions: {len(validation_relevant):,}")

def evaluate_validation(name, recommend_for_user):
    lists = {
        user_id: recommend_for_user(user_id, validation_seen.get(user_id, set()))[:RANK_K]
        for user_id in validation_users
    }
    scores = ranking_metrics(lists, validation_relevant_by_user)
    diversity_values = []
    for items in lists.values():
        genre_sets = [genres_by_movie.get(int(movie_id), set()) for movie_id in items]
        distances = []
        for left in range(len(genre_sets)):
            for right in range(left + 1, len(genre_sets)):
                union = genre_sets[left] | genre_sets[right]
                if union:
                    distances.append(1 - len(genre_sets[left] & genre_sets[right]) / len(union))
        if distances:
            diversity_values.append(float(np.mean(distances)))
    scores["Intra-list genre diversity"] = float(np.mean(diversity_values)) if diversity_values else 0.0
    return {"model": name, **scores}

validation_results = []
count_model = PopularityRecommender().fit(tuning_train, movies, min_ratings=20)
validation_results.append(evaluate_validation(
    "pop_count_min20",
    lambda user_id, seen: count_model.recommend(RANK_K, exclude_movie_ids=seen)["movieId"].tolist(),
))

for min_support in (5, 20, 50):
    for prior_weight in (10, 40):
        model = PopularityRecommender().fit(
            tuning_train, movies, min_ratings=min_support,
            score_mode="bayesian", prior_weight=prior_weight,
        )
        name = f"pop_bayesian_min{min_support}_prior{prior_weight}"
        validation_results.append(evaluate_validation(
            name,
            lambda user_id, seen, fitted=model: fitted.recommend(
                RANK_K, exclude_movie_ids=seen
            )["movieId"].tolist(),
        ))

for shrinkage in (0, 10, 50):
    model = ItemItemCollaborativeRecommender().fit(tuning_train, movies, shrinkage=shrinkage)
    name = f"item_item_shrink{shrinkage}"
    validation_results.append(evaluate_validation(
        name,
        lambda user_id, seen, fitted=model: fitted.recommend_for_user(
            user_id, n=RANK_K
        )["movieId"].tolist(),
    ))

validation_tuning = pd.DataFrame(validation_results).sort_values(
    [f"NDCG@{RANK_K}", "Catalog coverage@10", "Intra-list genre diversity"],
    ascending=[False, False, False],
).reset_index(drop=True)
display(validation_tuning)
validation_tuning.to_csv(OUTPUT_DIR / "baseline_validation_tuning.csv", index=False)
selected = validation_tuning.iloc[0]
print(f"Validation-selected configuration: {selected['model']}")
print(f"Validation NDCG@10: {selected[f'NDCG@{RANK_K}']:.4f}")
print(f"Saved: {OUTPUT_DIR / 'baseline_validation_tuning.csv'}")


Tuning train rows: 64,081; validation rows: 16,338
Validation users: 592; relevant interactions: 7,415


,model,Precision@10,Recall@10,NDCG@10,Catalog coverage@10,Users evaluated,Intra-list genre diversity
0,item_item_shrink50,0.077027,0.086368,0.110324,0.051016,592,0.767155
1,item_item_shrink10,0.072973,0.080400,0.103166,0.073086,592,0.772243
2,pop_count_min20,0.056926,0.075173,0.082292,0.007904,592,0.797795
3,pop_bayesian_min5_prior40,0.052872,0.063810,0.069401,0.005338,592,0.765109
4,pop_bayesian_min20_prior40,0.052872,0.063810,0.069401,0.005338,592,0.765109
5,pop_bayesian_min50_prior40,0.052703,0.063766,0.069293,0.005338,592,0.764817
6,pop_bayesian_min50_prior10,0.044426,0.048120,0.056919,0.005338,592,0.756107
7,pop_bayesian_min20_prior10,0.041723,0.046084,0.054752,0.004824,592,0.760777
8,pop_bayesian_min5_prior10,0.041385,0.046025,0.054520,0.004311,592,0.761752
9,item_item_shrink0,0.031588,0.043199,0.051065,0.176452,592,0.770211


Validation-selected configuration: item_item_shrink50
Validation NDCG@10: 0.1103
Saved: D:\Tastera\Tastera\data\processed\baseline_validation_tuning.csv


## 7. Interpretation and next steps

- MAE/RMSE compare rating prediction; Precision/Recall/NDCG compare top-K ranking.
- Hyperparameters are selected using validation only. The earlier reported test period was inspected, so it is not used to validate the selected configuration again.
- Compare relevance with catalog coverage and intra-list genre diversity; no single metric captures recommendation quality.
- MovieLens Latest Small is historical benchmark data, so results do not establish live-user quality.
- A fresh final evaluation needs new, later user interactions before production integration.


In [8]:
print("Saved output files:")
for path in sorted(OUTPUT_DIR.glob("baseline_*")):
    print(f"- {path.name} ({path.stat().st_size:,} bytes)")


Saved output files:
- baseline_content_top20.csv (1,517 bytes)
- baseline_popularity_top20.csv (1,912 bytes)
- baseline_ranking_metrics.csv (375 bytes)
- baseline_rating_metrics.csv (138 bytes)
- baseline_validation_tuning.csv (1,379 bytes)
